In [1]:
import random
import json
from datasets import Dataset

In [2]:
actions = ["move", "shift", "slide", "drag", "translate"]
shapes = ["circle", "circles", "line", "lines", "zigzag", "zigzags", "contact", "contacts"]
directions = ["left", "right", "up", "down"]
distance_range = list(range(1, 21))

instruction_templates = [
    "please {action} the {shape} {distance} pixels to the {direction}.",
    "can you {action} the {shape} by {distance} pixels {direction}?",
    "i need you to {action} the {shape} {distance} pixels toward the {direction}.",
    "{action} the {shape} {distance} pixels to the {direction}.",
    "kindly {action} the {shape} {distance} pixels to the {direction}.",
    "move the {shape} {distance} to the {direction}.",
    "make the {shape} go {distance} pixels to the {direction}.",
    "i want the {shape} to be {action}ed {distance} pixels to the {direction}.",
    "could you {action} the {shape} {distance} px {direction}?",
    "i'd like the {shape} to move {direction} by {distance} pixels.",
    "please perform a {action} of the {shape} by {distance} pixels {direction}.",
    "apply a {action} on the {shape} going {direction} for {distance} pixels.",
    "shift the position of the {shape} {distance} pixels to the {direction}.",
    "{action} {distance} pixels {direction} for the {shape}.",
    "slide the {shape} in the {direction} direction by {distance} pixels.",
    "drag the {shape} by {distance} pixels in the {direction} direction.",
    "translate the {shape} {direction} by {distance} pixels.",
    "move the {shape} exactly {distance} pixels to the {direction}.",
    "please nudge the {shape} {direction} by {distance} pixels.",
    "give the {shape} a {distance}-pixel {direction} shift.",
]

In [3]:
def generate_example():
    action = random.choice(actions)
    shape = random.choice(shapes)
    direction = random.choice(directions)
    distance = random.choice(distance_range)

    instruction = random.choice(instruction_templates).format(
        action=action,
        shape=shape,
        direction=direction,
        distance=distance
    )

    output = {
        "action": "move",
        "shape": shape.rstrip("s"),  # normalize plural
        "direction": direction,
        "distance": distance
    }

    return {"input": instruction, "output": json.dumps(output)}

In [4]:
num_examples = 100
examples = [generate_example() for _ in range(num_examples)]

In [5]:
dataset = Dataset.from_list(examples)
dataset_split = dataset.train_test_split(test_size=0.1)
dataset_split

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 90
    })
    test: Dataset({
        features: ['input', 'output'],
        num_rows: 10
    })
})

## Fine-tuning

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

Disabling PyTorch because PyTorch >= 2.1 is required but found 2.0.0


In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

In [ ]:
def preprocess(example):
    model_inputs = tokenizer(example["input"], max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(example["output"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_ds = dataset_split.map(preprocess, batched=True)

In [12]:
training_args = TrainingArguments(
    output_dir="./flan-t5-shape-parser",
    evaluation_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    save_total_limit=2,
    logging_dir="./logs",
    fp16=False  # set to True only if using GPU and want mixed precision
)

# Optional evaluation function
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Exact match score (basic)
    matches = [pred.strip() == label.strip() for pred, label in zip(decoded_preds, decoded_labels)]
    acc = sum(matches) / len(matches)
    return {"exact_match": acc}



trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer),
    compute_metrics=compute_metrics
)

TypeError: Accelerator.__init__() got an unexpected keyword argument 'dispatch_batches'

In [ ]:
trainer.train()